DataSource 

https://www.manythings.org/anki/
#### from English to arabic 

In [2]:
import numpy as np
import keras
import os
from pathlib import Path

In [3]:
DATA_PATH=r'D:\mena\documents\CS_AI\academicCourse\Software\AI_field\NLP\MachineTranslation\ara-eng\ara.txt'

In [ ]:
# DATA_PATH =r'D:\mena\documents\CS_AI\academicCourse\Software\AI_field\NLP\MachineTranslation\ara-eng\ara.txt'

In [4]:
batch_size = 64  # Batch size for training.
epochs = 100  # Number of epochs to train for.
latent_dim = 256  # Latent dimensionality of the encoding space.
num_samples = 10000  # Number of samples to train on.

In [5]:
with open(DATA_PATH, "r", encoding="utf-8") as f:
    lines = f.read().split("\n")
    print(len(lines))

12524


In [6]:
lines[:5]

['Hi.\tمرحبًا.\tCC-BY 2.0 (France) Attribution: tatoeba.org #538123 (CM) & #629296 (Samer)',
 'Run!\tاركض!\tCC-BY 2.0 (France) Attribution: tatoeba.org #906328 (papabear) & #1245450 (saeb)',
 'Duck!\tاخفض رأسك!\tCC-BY 2.0 (France) Attribution: tatoeba.org #280158 (CM) & #9036391 (KeEichi)',
 'Duck!\tاخفضي رأسك!\tCC-BY 2.0 (France) Attribution: tatoeba.org #280158 (CM) & #9036392 (KeEichi)',
 'Duck!\tاخفضوا رؤوسكم!\tCC-BY 2.0 (France) Attribution: tatoeba.org #280158 (CM) & #9036393 (KeEichi)']

In [7]:
input_texts = []
target_texts = []
input_characters = set()
target_characters = set()

In [8]:
with open(DATA_PATH, "r", encoding="utf-8") as f:
    lines = f.read().split("\n")
for line in lines[: min(num_samples, len(lines) - 1)]:
    input_text, target_text, _ = line.split("\t")
    target_text = "\t" + target_text + "\n"
    input_texts.append(input_text)
    target_texts.append(target_text)
    for char in input_text:
        if char not in input_characters:
            input_characters.add(char)
    for char in target_text:
        if char not in target_characters:
            target_characters.add(char)          

In [24]:
print(input_texts[:5])
print(target_texts[:5])
print(input_characters)
print(target_characters)

['Hi.', 'Run!', 'Duck!', 'Duck!', 'Duck!']
['\tمرحبًا.\n', '\tاركض!\n', '\tاخفض رأسك!\n', '\tاخفضي رأسك!\n', '\tاخفضوا رؤوسكم!\n']
{'6', 'N', 'g', 'z', 'n', 'i', 'w', 'j', 'V', ' ', 'y', 'd', 'J', ',', 'I', 'm', 'P', 't', 'b', '.', 'U', 'h', 'u', ':', '-', 'q', 'F', '3', '0', 'o', 'M', '2', 'D', 'v', '"', 'R', 'Y', 'x', '4', 'X', 'e', 'r', 'G', 's', 'B', 'L', '9', '7', 'K', '8', '!', 'W', 'O', "'", '€', '1', 'E', 'A', 'p', 'f', 'H', 'a', 'C', 'S', '5', '?', 'T', 'Q', 'l', 'k', 'c'}
{'ٌ', 'ث', '۰', ',', 'خ', 'ح', 'P', 'ل', '-', '0', '2', '٥', 'ذ', 'ّ', '٠', 'ج', 'م', '7', 'ش', '\\', 'ؤ', 'ب', 'ط', 'د', 'ن', '\u200d', '(', '٨', '6', '٤', 'i', 'ٱ', '؟', 'ـ', '\n', '/', 'ق', 'ر', 'x', '٢', '\u200f', 'ی', '9', '!', 'س', 'ه', 'ِ', 'غ', '١', '?', 'ت', 'ء', 'ً', '۹', 'إ', 'ئ', ':', '\t', 'F', '3', 'َ', '٩', ')', '"', 'ْ', 'و', 'ض', 'ة', 'r', 'أ', '٣', '5', 'ٍ', 'ي', ' ', 'ك', 'ص', '۱', '.', 'ا', 'o', 'M', 'ف', 'ع', 'e', 'آ', 'ز', 'ُ', 'ظ', '1', 'ى', 'f', '٦', '،'}


In [9]:
input_characters = sorted(list(input_characters))
target_characters = sorted(list(target_characters))
num_encoder_tokens = len(input_characters)
num_decoder_tokens = len(target_characters)
max_encoder_seq_length = max([len(txt) for txt in input_texts])
max_decoder_seq_length = max([len(txt) for txt in target_texts])

In [28]:
print("Number of samples:", len(input_texts))
print("Number of unique input characters:", num_encoder_tokens)
print("Number of unique output Characters:", num_decoder_tokens)
print("Max sequence length for inputs Tokens:", max_encoder_seq_length)
print("Max sequence length for outputs Tokens :", max_decoder_seq_length)

Number of samples: 10000
Number of unique input characters: 71
Number of unique output Characters: 94
Max sequence length for inputs Tokens: 36
Max sequence length for outputs Tokens : 54


In [10]:
input_token_index = dict([(char, i) for i, char in enumerate(input_characters)])
target_token_index = dict([(char, i) for i, char in enumerate(target_characters)])

encoder_input_data = np.zeros(
    (len(input_texts), max_encoder_seq_length, num_encoder_tokens),
    dtype="float32",
)
decoder_input_data = np.zeros(
    (len(input_texts), max_decoder_seq_length, num_decoder_tokens),
    dtype="float32",
)
decoder_target_data = np.zeros(
    (len(input_texts), max_decoder_seq_length, num_decoder_tokens),
    dtype="float32",
)

In [51]:
input_token_index

{' ': 0,
 '!': 1,
 '"': 2,
 "'": 3,
 ',': 4,
 '-': 5,
 '.': 6,
 '0': 7,
 '1': 8,
 '2': 9,
 '3': 10,
 '4': 11,
 '5': 12,
 '6': 13,
 '7': 14,
 '8': 15,
 '9': 16,
 ':': 17,
 '?': 18,
 'A': 19,
 'B': 20,
 'C': 21,
 'D': 22,
 'E': 23,
 'F': 24,
 'G': 25,
 'H': 26,
 'I': 27,
 'J': 28,
 'K': 29,
 'L': 30,
 'M': 31,
 'N': 32,
 'O': 33,
 'P': 34,
 'Q': 35,
 'R': 36,
 'S': 37,
 'T': 38,
 'U': 39,
 'V': 40,
 'W': 41,
 'X': 42,
 'Y': 43,
 'a': 44,
 'b': 45,
 'c': 46,
 'd': 47,
 'e': 48,
 'f': 49,
 'g': 50,
 'h': 51,
 'i': 52,
 'j': 53,
 'k': 54,
 'l': 55,
 'm': 56,
 'n': 57,
 'o': 58,
 'p': 59,
 'q': 60,
 'r': 61,
 's': 62,
 't': 63,
 'u': 64,
 'v': 65,
 'w': 66,
 'x': 67,
 'y': 68,
 'z': 69,
 '€': 70}

In [52]:
target_token_index

{'\t': 0,
 '\n': 1,
 ' ': 2,
 '!': 3,
 '"': 4,
 '(': 5,
 ')': 6,
 ',': 7,
 '-': 8,
 '.': 9,
 '/': 10,
 '0': 11,
 '1': 12,
 '2': 13,
 '3': 14,
 '5': 15,
 '6': 16,
 '7': 17,
 '9': 18,
 ':': 19,
 '?': 20,
 'F': 21,
 'M': 22,
 'P': 23,
 '\\': 24,
 'e': 25,
 'f': 26,
 'i': 27,
 'o': 28,
 'r': 29,
 'x': 30,
 '،': 31,
 '؟': 32,
 'ء': 33,
 'آ': 34,
 'أ': 35,
 'ؤ': 36,
 'إ': 37,
 'ئ': 38,
 'ا': 39,
 'ب': 40,
 'ة': 41,
 'ت': 42,
 'ث': 43,
 'ج': 44,
 'ح': 45,
 'خ': 46,
 'د': 47,
 'ذ': 48,
 'ر': 49,
 'ز': 50,
 'س': 51,
 'ش': 52,
 'ص': 53,
 'ض': 54,
 'ط': 55,
 'ظ': 56,
 'ع': 57,
 'غ': 58,
 'ـ': 59,
 'ف': 60,
 'ق': 61,
 'ك': 62,
 'ل': 63,
 'م': 64,
 'ن': 65,
 'ه': 66,
 'و': 67,
 'ى': 68,
 'ي': 69,
 'ً': 70,
 'ٌ': 71,
 'ٍ': 72,
 'َ': 73,
 'ُ': 74,
 'ِ': 75,
 'ّ': 76,
 'ْ': 77,
 '٠': 78,
 '١': 79,
 '٢': 80,
 '٣': 81,
 '٤': 82,
 '٥': 83,
 '٦': 84,
 '٨': 85,
 '٩': 86,
 'ٱ': 87,
 'ی': 88,
 '۰': 89,
 '۱': 90,
 '۹': 91,
 '\u200d': 92,
 '\u200f': 93}

In [33]:
print(encoder_input_data.shape)
print(decoder_input_data.shape)
print(decoder_target_data.shape)

(10000, 36, 71)
(10000, 54, 94)
(10000, 54, 94)


In [34]:
print(input_token_index)
print(target_token_index)

{' ': 0, '!': 1, '"': 2, "'": 3, ',': 4, '-': 5, '.': 6, '0': 7, '1': 8, '2': 9, '3': 10, '4': 11, '5': 12, '6': 13, '7': 14, '8': 15, '9': 16, ':': 17, '?': 18, 'A': 19, 'B': 20, 'C': 21, 'D': 22, 'E': 23, 'F': 24, 'G': 25, 'H': 26, 'I': 27, 'J': 28, 'K': 29, 'L': 30, 'M': 31, 'N': 32, 'O': 33, 'P': 34, 'Q': 35, 'R': 36, 'S': 37, 'T': 38, 'U': 39, 'V': 40, 'W': 41, 'X': 42, 'Y': 43, 'a': 44, 'b': 45, 'c': 46, 'd': 47, 'e': 48, 'f': 49, 'g': 50, 'h': 51, 'i': 52, 'j': 53, 'k': 54, 'l': 55, 'm': 56, 'n': 57, 'o': 58, 'p': 59, 'q': 60, 'r': 61, 's': 62, 't': 63, 'u': 64, 'v': 65, 'w': 66, 'x': 67, 'y': 68, 'z': 69, '€': 70}
{'\t': 0, '\n': 1, ' ': 2, '!': 3, '"': 4, '(': 5, ')': 6, ',': 7, '-': 8, '.': 9, '/': 10, '0': 11, '1': 12, '2': 13, '3': 14, '5': 15, '6': 16, '7': 17, '9': 18, ':': 19, '?': 20, 'F': 21, 'M': 22, 'P': 23, '\\': 24, 'e': 25, 'f': 26, 'i': 27, 'o': 28, 'r': 29, 'x': 30, '،': 31, '؟': 32, 'ء': 33, 'آ': 34, 'أ': 35, 'ؤ': 36, 'إ': 37, 'ئ': 38, 'ا': 39, 'ب': 40, 'ة': 41

In [11]:
for i, (input_text, target_text) in enumerate(zip(input_texts, target_texts)):
    for t, char in enumerate(input_text):
        encoder_input_data[i, t, input_token_index[char]] = 1.0
    encoder_input_data[i, t + 1 :, input_token_index[" "]] = 1.0
    for t, char in enumerate(target_text):
        # decoder_target_data is ahead of decoder_input_data by one timestep
        decoder_input_data[i, t, target_token_index[char]] = 1.0
        if t > 0:
            # decoder_target_data will be ahead by one timestep
            # and will not include the start character.
            decoder_target_data[i, t - 1, target_token_index[char]] = 1.0
    decoder_input_data[i, t + 1 :, target_token_index[" "]] = 1.0
    decoder_target_data[i, t:, target_token_index[" "]] = 1.0

In [12]:
encoder_inputs = keras.Input(shape=(None, num_encoder_tokens)) #n_unique characters in the encoder -> here the emglish 
encoder = keras.layers.LSTM(latent_dim, return_state=True) # n nuerons
encoder_outputs, state_h, state_c = encoder(encoder_inputs) # cell sate , hidden state 
# do not care about encoder_outputs_now
encoder_states = [state_h, state_c]

# Set up the decoder, using `encoder_states` as initial state.
decoder_inputs = keras.Input(shape=(None, num_decoder_tokens)) # n_unique caharcters for the decoder

decoder_lstm = keras.layers.LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_inputs, initial_state=encoder_states)
decoder_dense = keras.layers.Dense(num_decoder_tokens, activation="softmax") # the next word prediction
decoder_outputs = decoder_dense(decoder_outputs)


# Define the model that will turn
# `encoder_input_data` & `decoder_input_data` into `decoder_target_data`
model = keras.Model([encoder_inputs, decoder_inputs], decoder_outputs)

In [22]:
model.compile(
    optimizer="rmsprop", loss="categorical_crossentropy", metrics=["accuracy"]
)
model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    batch_size=batch_size,
    epochs=5,
    validation_split=0.2,
)



#model.save("s2s_model.keras")

Epoch 1/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 17s 116ms/step - accuracy: 0.9392 - loss: 0.2163 - val_accuracy: 0.7120 - val_loss: 1.5914
Epoch 2/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 14s 114ms/step - accuracy: 0.9434 - loss: 0.2049 - val_accuracy: 0.7109 - val_loss: 1.5985
Epoch 3/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 15s 120ms/step - accuracy: 0.9441 - loss: 0.2029 - val_accuracy: 0.7118 - val_loss: 1.6039
Epoch 4/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 15s 120ms/step - accuracy: 0.9443 - loss: 0.2018 - val_accuracy: 0.7115 - val_loss: 1.6103
Epoch 5/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 15s 117ms/step - accuracy: 0.9447 - loss: 0.2009 - val_accuracy: 0.7103 - val_loss: 1.6213


In [23]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None, 71)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, None, 94)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 256),     │    335,872 │ input_layer[0][0] │
│                     │ (None, 256),      │            │                   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, None,     │    359,424 │ input_layer_1[0]… │
│                     │ 256), (None,      │            │ lstm[0][1],       │
│                     │ 256), (None,      │            │ lstm[0][2]        │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None, 94)  │     24,158 │ lstm_1[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,438,910 (5.49 MB)

 Trainable params: 719,454 (2.74 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 719,456 (2.74 MB)

In [19]:
encoder_inputs = model.input[0]  # input_1
encoder_outputs, state_h_enc, state_c_enc = model.layers[2].output  # lstm_1
encoder_states = [state_h_enc, state_c_enc]
encoder_model = keras.Model(encoder_inputs, encoder_states)

decoder_inputs = model.input[1]  # input_2
decoder_state_input_h = keras.Input(shape=(latent_dim,))
decoder_state_input_c = keras.Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]
decoder_lstm = model.layers[3]
decoder_outputs, state_h_dec, state_c_dec = decoder_lstm(
    decoder_inputs, initial_state=decoder_states_inputs
)
decoder_states = [state_h_dec, state_c_dec]
decoder_dense = model.layers[4]
decoder_outputs = decoder_dense(decoder_outputs)
decoder_model = keras.Model(
    [decoder_inputs] + decoder_states_inputs, [decoder_outputs] + decoder_states
)

# Reverse-lookup token index to decode sequences back to
# something readable.
reverse_input_char_index = dict((i, char) for char, i in input_token_index.items())
reverse_target_char_index = dict((i, char) for char, i in target_token_index.items())

In [23]:
def decode_sequence(input_seq):
    # Encode the input as state vectors.
    states_value = encoder_model.predict(input_seq, verbose=0)

    # Generate empty target sequence of length 1.
    target_seq = np.zeros((1, 1, num_decoder_tokens))
    # Populate the first character of target sequence with the start character.
    target_seq[0, 0, target_token_index["\t"]] = 1.0

    # Sampling loop for a batch of sequences
    # (to simplify, here we assume a batch of size 1).
    stop_condition = False
    decoded_sentence = ""
    while not stop_condition:
        output_tokens, h, c = decoder_model.predict(
            [target_seq] + states_value, verbose=0
        )

        # Sample a token
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_char = reverse_target_char_index[sampled_token_index]
        decoded_sentence += sampled_char

        # Exit condition: either hit max length
        # or find stop character.
        if sampled_char == "\n" or len(decoded_sentence) > max_decoder_seq_length:
            stop_condition = True

        # Update the target sequence (of length 1).
        target_seq = np.zeros((1, 1, num_decoder_tokens))
        target_seq[0, 0, sampled_token_index] = 1.0

        # Update states
        states_value = [h, c]
    return decoded_sentence

In [47]:
for seq_index in range(20):
    # Take one sequence (part of the training set)
    # for trying out decoding.
    input_seq = encoder_input_data[seq_index : seq_index + 1]
    decoded_sentence = decode_sequence(input_seq)
    print("-")
    print("Input sentence:", input_texts[seq_index])
    print("Decoded sentence:", decoded_sentence)

-
Input sentence: Hi.
Decoded sentence: مرحبًا.

-
Input sentence: Run!
Decoded sentence: اركض!

-
Input sentence: Duck!
Decoded sentence: اخفضوا رؤوسكم!

-
Input sentence: Duck!
Decoded sentence: اخفضوا رؤوسكم!

-
Input sentence: Duck!
Decoded sentence: اخفضوا رؤوسكم!

-
Input sentence: Help!
Decoded sentence: النجدة!

-
Input sentence: Jump!
Decoded sentence: اقفض!

-
Input sentence: Stop!
Decoded sentence: توقف !

-
Input sentence: Stop!
Decoded sentence: توقف !

-
Input sentence: Wait!
Decoded sentence: إنتظر

-
Input sentence: Go on.
Decoded sentence: استمر.

-
Input sentence: Go on.
Decoded sentence: استمر.

-
Input sentence: Hello!
Decoded sentence: مرحباً.

-
Input sentence: Hello.
Decoded sentence: مرحباً.

-
Input sentence: Hello.
Decoded sentence: مرحباً.

-
Input sentence: Hurry!
Decoded sentence: تعجّل!

-
Input sentence: Hurry!
Decoded sentence: تعجّل!

-
Input sentence: I see.
Decoded sentence: انا اري

-
Input sentence: I won!
Decoded sentence: أنا فُزت!

-
Input senten

In [24]:
def predict_sentence(input_sentence):
    # Tokenize and one-hot encode the input sequence
    input_seq = np.zeros((1, max_encoder_seq_length, num_encoder_tokens), dtype='float32')
    
    for t, char in enumerate(input_sentence):
        if t < max_encoder_seq_length:
            input_seq[0, t, input_token_index.get(char, 0)] = 1.0  # Use 0 if char is not found

    # Decode the sequence
    decoded_sentence = decode_sequence(input_seq)
    print("-")
    print("Input sentence:", input_sentence)
    print("Decoded sentence:", decoded_sentence)


In [25]:
predict_sentence("Relax")

-
Input sentence: Relax
Decoded sentence: هل تحب الماء الماء الماضي؟

